# Module 2 · Class 5 · Level 2
# Descriptive Statistics and Correlation — Homework-Style Notebook

This version is closer to the assignment format. It gives structure, but you must complete more code and write more interpretation yourself.

Core tasks:
1. Load the Superstore dataset.
2. Take a sample of 10 Sales values.
3. Calculate mean, variance, and standard deviation manually.
4. Verify with NumPy.
5. Analyze mean vs median.
6. Compute and interpret correlations between Sales, Profit, and Discount.

## 0. Setup

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 12

print("Setup complete.")

## 0.1 Dataset loading

Use **one** of the three loading options below.

**Option 1 — Preferred:** use `superstore_cleaned.csv` or `SampleSuperstore.csv` already uploaded to Colab.  
**Option 2 — Manual upload:** upload the CSV from your computer.  
**Option 3 — External source:** download the Superstore dataset from Kaggle, with a public CSV fallback.

Set `LOAD_MODE` to one of: `"preferred"`, `"manual_upload"`, `"external_source"`.

In [ ]:
LOAD_MODE = "preferred"  # "preferred", "manual_upload", or "external_source"

PREFERRED_PATHS = [
    "superstore_cleaned.csv",
    "SampleSuperstore.csv",
    "Sample - Superstore.csv",
    "/content/superstore_cleaned.csv",
    "/content/SampleSuperstore.csv",
    "/content/Sample - Superstore.csv",
]

KAGGLE_DATASET = "vivek468/superstore-dataset-final"
PUBLIC_FALLBACK_URL = "https://raw.githubusercontent.com/dsrscientist/dataset1/master/superstore.csv"


def read_csv_safely(path_or_url):
    """Try common encodings used by Superstore CSV files."""
    last_error = None
    for enc in ["utf-8", "latin-1", "ISO-8859-1"]:
        try:
            return pd.read_csv(path_or_url, encoding=enc)
        except Exception as e:
            last_error = e
    raise last_error


def find_csv_file(folder):
    csvs = glob.glob(os.path.join(folder, "**", "*.csv"), recursive=True)
    if not csvs:
        raise FileNotFoundError("No CSV file found in the downloaded dataset folder.")
    return csvs[0]


def load_superstore(load_mode=LOAD_MODE):
    if load_mode == "preferred":
        for path in PREFERRED_PATHS:
            if os.path.exists(path):
                print(f"Loading preferred local file: {path}")
                return read_csv_safely(path)
        raise FileNotFoundError(
            "Preferred file not found. Upload superstore_cleaned.csv / SampleSuperstore.csv, "
            "or set LOAD_MODE to 'manual_upload' or 'external_source'."
        )

    if load_mode == "manual_upload":
        from google.colab import files
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("No file uploaded.")
        filename = list(uploaded.keys())[0]
        print(f"Loading uploaded file: {filename}")
        return read_csv_safely(filename)

    if load_mode == "external_source":
        try:
            try:
                import kagglehub
            except ImportError:
                print("Installing kagglehub...")
                get_ipython().run_line_magic("pip", "-q install kagglehub")
                import kagglehub
            dataset_path = kagglehub.dataset_download(KAGGLE_DATASET)
            csv_path = find_csv_file(dataset_path)
            print(f"Loading Kaggle dataset file: {csv_path}")
            return read_csv_safely(csv_path)
        except Exception as e:
            print(f"Kaggle loading failed: {e}")
            print("Trying public CSV fallback...")
            return read_csv_safely(PUBLIC_FALLBACK_URL)

    raise ValueError("LOAD_MODE must be 'preferred', 'manual_upload', or 'external_source'.")


def normalize_column_name(name):
    return str(name).lower().replace(" ", "").replace("_", "").replace("-", "")


def find_column(df, possible_names):
    lookup = {normalize_column_name(c): c for c in df.columns}
    for name in possible_names:
        key = normalize_column_name(name)
        if key in lookup:
            return lookup[key]
    # fallback: substring match
    for c in df.columns:
        c_norm = normalize_column_name(c)
        if any(normalize_column_name(name) in c_norm for name in possible_names):
            return c
    raise KeyError(f"Could not find any of these columns: {possible_names}. Available columns: {list(df.columns)}")


df = load_superstore()

# Standard column references used below.
SALES_COL = find_column(df, ["Sales"])
PROFIT_COL = find_column(df, ["Profit"])
DISCOUNT_COL = find_column(df, ["Discount"])
QUANTITY_COL = find_column(df, ["Quantity"])

print(f"Loaded dataset: {df.shape[0]} rows, {df.shape[1]} columns")
print("Core columns:", SALES_COL, PROFIT_COL, DISCOUNT_COL, QUANTITY_COL)
df.head()

## Task 1 — Sample 10 Sales values

Use the `Sales` column and `random_state=42` so your result is reproducible.

In [ ]:
# TODO: take a sample of 10 Sales values from df.
sample = df[SALES_COL].dropna().sample(10, random_state=42).values

print("Sample values:", sample)
print("Sample size:", len(sample))

## Task 2 — Manual mean

Formula: mean = sum of values / number of values.

In [ ]:
# TODO: calculate mean manually.
manual_mean = sum(sample) / len(sample)

print(f"Manual mean: {manual_mean:.2f}")
print(f"NumPy mean:  {np.mean(sample):.2f}")
print("Match:", np.isclose(manual_mean, np.mean(sample)))

## Task 3 — Manual variance

Formula: variance = average of squared differences from the mean.

Steps:
1. subtract the mean from each value;
2. square each difference;
3. average the squared differences.

In [ ]:
# TODO: calculate variance manually.
squared_diffs = [(x - manual_mean) ** 2 for x in sample]
manual_variance = sum(squared_diffs) / len(sample)

print(f"Manual variance: {manual_variance:.2f}")
print(f"NumPy variance:  {np.var(sample):.2f}")
print("Match:", np.isclose(manual_variance, np.var(sample)))

## Task 4 — Manual standard deviation

Formula: standard deviation = square root of variance.

In [ ]:
# TODO: calculate standard deviation manually.
manual_std = manual_variance ** 0.5

print(f"Manual std dev: {manual_std:.2f}")
print(f"NumPy std dev:  {np.std(sample):.2f}")
print("Match:", np.isclose(manual_std, np.std(sample)))

## Task 5 — Interpret descriptive statistics

Write 2–3 sentences for each question:

1. What does the mean tell you about this sample?
2. What does variance tell you that the mean does not?
3. If standard deviation is large relative to the mean, what does that imply?

**Your answer:**

...

## Task 6 — Mean vs median on the full Sales column

Compare mean and median. Then plot the distribution.

In [ ]:
sales = df[SALES_COL].dropna()

mean_sales = sales.mean()
median_sales = sales.median()

print(f"Mean Sales:   {mean_sales:.2f}")
print(f"Median Sales: {median_sales:.2f}")
print(f"Mean / Median: {mean_sales / median_sales:.2f}x")

plt.figure(figsize=(10, 5))
plt.hist(sales, bins=80, edgecolor="black", alpha=0.7)
plt.axvline(mean_sales, linestyle="--", linewidth=2, label="Mean")
plt.axvline(median_sales, linestyle="--", linewidth=2, label="Median")
plt.title("Sales Distribution")
plt.xlabel("Sales")
plt.ylabel("Frequency")
plt.legend()
plt.show()

## Task 7 — Interpret mean vs median

Answer:

1. Is the Sales distribution symmetric or skewed?
2. Which value would you report to a business stakeholder: mean or median?
3. Why?

**Your answer:**

...

## Task 8 — Correlation analysis

Compute correlations between `Sales`, `Profit`, and `Discount`.

In [ ]:
# TODO: compute individual correlations.
corr_sales_profit = df[SALES_COL].corr(df[PROFIT_COL])
corr_sales_discount = df[SALES_COL].corr(df[DISCOUNT_COL])
corr_discount_profit = df[DISCOUNT_COL].corr(df[PROFIT_COL])

print(f"Sales-Profit correlation:    {corr_sales_profit:.4f}")
print(f"Sales-Discount correlation:  {corr_sales_discount:.4f}")
print(f"Discount-Profit correlation: {corr_discount_profit:.4f}")

In [ ]:
# TODO: generate the full correlation matrix.
corr_matrix = df[[SALES_COL, PROFIT_COL, DISCOUNT_COL]].corr()
corr_matrix

In [ ]:
# Optional but recommended: heatmap.
plt.figure(figsize=(7, 5))
sns.heatmap(corr_matrix, annot=True, cmap="RdBu_r", center=0, vmin=-1, vmax=1)
plt.title("Correlation Matrix: Sales, Profit, Discount")
plt.show()

In [ ]:
# Optional bonus: scatter plot.
plt.figure(figsize=(8, 5))
plt.scatter(df[SALES_COL], df[PROFIT_COL], alpha=0.3, s=10)
plt.title("Sales vs Profit")
plt.xlabel("Sales")
plt.ylabel("Profit")
plt.show()

## Task 9 — Interpret correlations

Answer:

1. Which pair is most strongly correlated?
2. Is any relationship negative? What does negative correlation mean here?
3. If two features have correlation 0.95, should you include both in a model? Why or why not?
4. Does correlation prove causation?

**Your answer:**

...

## Submission checklist

Before submitting:

- [ ] All code cells executed.
- [ ] Manual calculations match NumPy results.
- [ ] Mean vs median explained.
- [ ] Correlation matrix printed.
- [ ] At least one correlation visualization included.
- [ ] Interpretation answers written in markdown cells.